# 00 · Make the server logs
- A Java web shop, `shop-app`: **Apache httpd → Tomcat (Spring Boot) → PostgreSQL**, plus a payment gateway
- 3 days of logs in 5 files, with **5 incidents planted at known times**
- The answer key (`incidents.json`) lets us mark the chatbot right or wrong later
- A small knowledge base (runbooks, post-mortems) for the RAG version

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'          # models come from the local cache only
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.max_colwidth', 120)
from logbot.make_data import generate
counts = generate()
counts

{'apache/access_log': 28647,
 'apache/error_log': 30,
 'app/application.log': 419,
 'tomcat/catalina.out': 8,
 'deploys.log': 3}

## The five log files

In [2]:
from logbot import LOGS
for name in counts:
    print(f'--- {name}')
    print('\n'.join((LOGS / name).read_text().splitlines()[:3]))

--- apache/access_log
10.0.8.22 - - [13/Oct/2026:00:00:42 +0000] "POST /api/orders HTTP/1.1" 201 2806 "-" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_5)" 104ms
10.0.4.159 - - [13/Oct/2026:00:01:24 +0000] "GET /api/products HTTP/1.1" 200 2134 "-" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_5)" 20ms
10.0.0.1 - - [13/Oct/2026:00:01:37 +0000] "GET /health HTTP/1.1" 200 3551 "-" "kube-probe/1.30" 3ms
--- apache/error_log
[Tue Oct 13 02:20:01.000 2026] [core:info] [pid 2211] [client 10.0.2.63:42146] AH00128: File does not exist: /var/www/html/favicon.ico
[Tue Oct 13 09:58:11.000 2026] [core:info] [pid 2211] [client 10.0.7.14:44183] AH00128: File does not exist: /var/www/html/favicon.ico
[Tue Oct 13 11:51:58.000 2026] [core:info] [pid 2211] [client 10.0.5.228:55930] AH00128: File does not exist: /var/www/html/favicon.ico
--- app/application.log
2026-10-13 00:52:49.000  INFO 4512 --- [   nio-8080-exec-3] c.shop.catalog.ProductCache              : Cache refresh done: 9653 products in 700 ms
2026-10-13 02

## A Java stack trace: one error, several lines

In [3]:
text = (LOGS / 'app/application.log').read_text()
i = text.index('SQLTransientConnectionException')
print(text[text.rindex('\n', 0, i) + 1: i + 330])

2026-10-13 10:21:45.716 ERROR 4512 --- [   nio-8080-exec-2] c.shop.orders.OrderService               : java.sql.SQLTransientConnectionException: HikariPool-1 - Connection is not available, request timed out after 30000ms.
	at com.zaxxer.hikari.pool.HikariPool.createTimeoutException(HikariPool.java:696)
	at com.zaxxer.hikari.pool.HikariPool.getConnection(HikariPool.java:181)
	at c.shop.orders.OrderRepository.findById(OrderRepository.java:4


## The answer key

In [4]:
import json
from logbot import DATA
key = pd.DataFrame(json.loads((DATA / 'incidents.json').read_text()))
key[['id', 'title', 'start', 'end', 'endpoint']]

,id,title,start,end,endpoint
0,db-pool,Database connection pool exhausted,2026-10-13T10:20:00+00:00,2026-10-13T10:45:00+00:00,/api/orders
1,oom,Memory leak → OutOfMemoryError → Tomcat restart,2026-10-14T08:00:00+00:00,2026-10-14T11:04:00+00:00,all
2,gateway,Slow payment gateway,2026-10-14T16:10:00+00:00,2026-10-14T16:35:00+00:00,/api/checkout
3,bad-deploy,Bad deploy v1.4.2,2026-10-15T14:00:00+00:00,2026-10-15T14:40:00+00:00,/api/users
4,brute-force,Login brute force,2026-10-15T22:05:00+00:00,2026-10-15T22:20:00+00:00,/api/login


## The knowledge base

In [6]:
from logbot import KB
sorted(str(p.relative_to(KB)) for p in KB.rglob('*.md'))

['architecture.md',
 'postmortems/2026-08-14-oom.md',
 'postmortems/2026-09-02-db-pool.md',
 'runbooks/bad-deploy-rollback.md',
 'runbooks/brute-force-login.md',
 'runbooks/hikari-pool.md',
 'runbooks/java-heap-oom.md',
 'runbooks/payment-gateway-timeouts.md']

In [6]:
assert counts['apache/access_log'] > 25_000 and len(key) == 5
assert (KB / 'runbooks/java-heap-oom.md').exists()
print('✓ logs, answer key and knowledge base written')

✓ logs, answer key and knowledge base written
